# Text Image Deblurring using Transfer Learning with Pretrained CNN Models

This notebook trains a deep learning model for text image deblurring using pretrained VGG16/ResNet50 as encoder in an autoencoder architecture.

**Key Features:**
- Transfer learning with VGG16/ResNet50
- Custom decoder for image reconstruction
- PSNR and SSIM metrics
- Model checkpointing and visualization

**Compatible with:** Kaggle, Google Colab, Local GPU

## 1. Setup and Imports

In [ ]:
# Check GPU availability
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"Built with CUDA: {tf.test.is_built_with_cuda()}")

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# TensorFlow/Keras
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# For metrics
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

print("All libraries imported successfully!")

## 2. Configuration

Set up paths and hyperparameters

## 2. Configuration

Set up paths and hyperparameters

**Dataset Info:**
- You have: `text-deblurring-dataset-with-psf-for-ocr`
- Contains: BMVC_image_data (blurred), BMVC_OCR_test_data (original)
- Also available: BMVC_image_quality_test_data

**Quick Setup:**
1. For Kaggle: Use the paths below (already configured)
2. For Local: Uncomment the LOCAL_PATHS section

In [ ]:
# Environment Detection and Path Setup
import os

# Detect environment
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
IS_LOCAL = not (IS_KAGGLE or IS_COLAB)

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")

# Set paths based on environment
if IS_KAGGLE:
    # Kaggle paths - your dataset is already added!
    BLUR_DIR = '/kaggle/input/text-deblurring-dataset-with-psf-for-ocr/BMVC_image_data'
    ORIG_DIR = '/kaggle/input/text-deblurring-dataset-with-psf-for-ocr/BMVC_OCR_test_data'
    MODEL_SAVE_PATH = '/kaggle/working/saved_models'
    print("✅ Using Kaggle dataset paths")
elif IS_COLAB:
    # Colab paths
    BLUR_DIR = '/content/data/blur'
    ORIG_DIR = '/content/data/orig'
    MODEL_SAVE_PATH = '/content/saved_models'
    print("✅ Using Colab paths")
else:
    # Local paths
    BLUR_DIR = r'R:\AI Image Deblurring\text_deblurring_pretrained\data\blur'
    ORIG_DIR = r'R:\AI Image Deblurring\text_deblurring_pretrained\data\orig'
    MODEL_SAVE_PATH = r'R:\AI Image Deblurring\text_deblurring_pretrained\saved_models'
    print("✅ Using local paths")

print(f"\nDataset paths:")
print(f"  Blur: {BLUR_DIR}")
print(f"  Orig: {ORIG_DIR}")
print(f"  Save: {MODEL_SAVE_PATH}")

In [ ]:
# Configuration
CONFIG = {
    # Paths (automatically set based on environment)
    'BLUR_DIR': BLUR_DIR,
    'ORIG_DIR': ORIG_DIR,
    'MODEL_SAVE_PATH': MODEL_SAVE_PATH,
    
    # Model selection: 'vgg16' or 'resnet50'
    'MODEL_TYPE': 'vgg16',
    
    # Image parameters
    'IMG_HEIGHT': 256,
    'IMG_WIDTH': 256,
    'IMG_CHANNELS': 3,
    
    # Training parameters
    'BATCH_SIZE': 16,
    'EPOCHS': 50,
    'LEARNING_RATE': 0.001,
    'VALIDATION_SPLIT': 0.2,
    
    # Model parameters
    'FREEZE_ENCODER': True,  # Freeze pretrained encoder weights
    
    # For dummy data (if no dataset available)
    'USE_DUMMY_DATA': False,  # Set to True if you want to test with dummy data
    'NUM_DUMMY_SAMPLES': 200
}

# Create save directory
os.makedirs(CONFIG['MODEL_SAVE_PATH'], exist_ok=True)

print("\n" + "="*60)
print("CONFIGURATION SUMMARY")
print("="*60)
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print("="*60)

## 3. Data Loading and Preprocessing

In [ ]:
def load_and_preprocess_image(image_path, target_size):
    """
    Load and preprocess a single image.
    """
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    return img


def create_dummy_dataset(blur_dir, orig_dir, num_samples, img_size):
    """
    Create dummy blurred and sharp image pairs for testing.
    """
    os.makedirs(blur_dir, exist_ok=True)
    os.makedirs(orig_dir, exist_ok=True)
    
    print(f"Generating {num_samples} dummy image pairs...")
    
    for i in range(num_samples):
        # Create synthetic image with text-like patterns
        img = np.random.randint(200, 255, (*img_size, 3), dtype=np.uint8)
        
        # Add rectangular regions (simulating text)
        num_rects = np.random.randint(3, 8)
        for _ in range(num_rects):
            x1 = np.random.randint(0, img_size[1] - 50)
            y1 = np.random.randint(0, img_size[0] - 20)
            x2 = x1 + np.random.randint(30, 100)
            y2 = y1 + np.random.randint(10, 30)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 0), -1)
        
        # Save original
        orig_path = os.path.join(orig_dir, f"img_{i:04d}.png")
        cv2.imwrite(orig_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        
        # Create and save blurred version
        blurred = cv2.GaussianBlur(img, (15, 15), 0)
        blur_path = os.path.join(blur_dir, f"img_{i:04d}.png")
        cv2.imwrite(blur_path, cv2.cvtColor(blurred, cv2.COLOR_RGB2BGR))
    
    print(f"Dummy dataset created!")


def load_dataset(blur_dir, orig_dir, img_size):
    """
    Load all image pairs from directories.
    """
    blur_dir = Path(blur_dir)
    orig_dir = Path(orig_dir)
    
    # Get image files
    blur_files = sorted([f for f in blur_dir.glob('*') 
                        if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']])
    orig_files = sorted([f for f in orig_dir.glob('*') 
                        if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']])
    
    # Match by filename
    blur_dict = {f.stem: f for f in blur_files}
    orig_dict = {f.stem: f for f in orig_files}
    
    blur_images = []
    orig_images = []
    
    for name in blur_dict:
        if name in orig_dict:
            blur_img = load_and_preprocess_image(blur_dict[name], img_size)
            orig_img = load_and_preprocess_image(orig_dict[name], img_size)
            blur_images.append(blur_img)
            orig_images.append(orig_img)
    
    print(f"Loaded {len(blur_images)} image pairs")
    return np.array(blur_images), np.array(orig_images)


# Load or create dataset
img_size = (CONFIG['IMG_HEIGHT'], CONFIG['IMG_WIDTH'])

if CONFIG['USE_DUMMY_DATA']:
    print("Creating dummy dataset...")
    dummy_blur_dir = '/kaggle/working/data/blur'
    dummy_orig_dir = '/kaggle/working/data/orig'
    create_dummy_dataset(dummy_blur_dir, dummy_orig_dir, 
                        CONFIG['NUM_DUMMY_SAMPLES'], img_size)
    X, y = load_dataset(dummy_blur_dir, dummy_orig_dir, img_size)
else:
    print("Loading real dataset...")
    X, y = load_dataset(CONFIG['BLUR_DIR'], CONFIG['ORIG_DIR'], img_size)

print(f"\nDataset shape:")
print(f"  Blurred images (X): {X.shape}")
print(f"  Original images (y): {y.shape}")

### Visualize Sample Data

In [ ]:
# Visualize sample pairs
fig, axes = plt.subplots(3, 2, figsize=(10, 12))

for i in range(3):
    axes[i, 0].imshow(X[i])
    axes[i, 0].set_title(f"Blurred Image {i+1}")
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(y[i])
    axes[i, 1].set_title(f"Original Image {i+1}")
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_data.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sample visualization saved!")

### Train/Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=CONFIG['VALIDATION_SPLIT'],
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

## 4. Model Architecture

Build the pretrained CNN-based autoencoder

In [ ]:
def build_vgg16_autoencoder(input_shape, freeze_encoder=True):
    """
    Build VGG16-based autoencoder.
    """
    inputs = layers.Input(shape=input_shape, name='input_image')
    
    # Encoder: Pretrained VGG16
    vgg16_base = VGG16(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs
    )
    
    if freeze_encoder:
        for layer in vgg16_base.layers:
            layer.trainable = False
    
    encoder_output = vgg16_base.output
    
    # Decoder: Upsampling path
    x = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(encoder_output)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    outputs = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='VGG16_Autoencoder')
    return model


def build_resnet_autoencoder(input_shape, freeze_encoder=True):
    """
    Build ResNet50-based autoencoder.
    """
    inputs = layers.Input(shape=input_shape, name='input_image')
    
    # Encoder: Pretrained ResNet50
    resnet_base = ResNet50(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs
    )
    
    if freeze_encoder:
        for layer in resnet_base.layers:
            layer.trainable = False
    
    encoder_output = resnet_base.output
    
    # Decoder: Upsampling path
    x = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(encoder_output)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)
    
    outputs = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='ResNet50_Autoencoder')
    return model


# Build model
input_shape = (CONFIG['IMG_HEIGHT'], CONFIG['IMG_WIDTH'], CONFIG['IMG_CHANNELS'])

print(f"Building {CONFIG['MODEL_TYPE'].upper()} autoencoder...")

if CONFIG['MODEL_TYPE'] == 'vgg16':
    model = build_vgg16_autoencoder(input_shape, CONFIG['FREEZE_ENCODER'])
elif CONFIG['MODEL_TYPE'] == 'resnet50':
    model = build_resnet_autoencoder(input_shape, CONFIG['FREEZE_ENCODER'])
else:
    raise ValueError(f"Unknown model type: {CONFIG['MODEL_TYPE']}")

print(f"\nModel created successfully!")
print(f"Total parameters: {model.count_params():,}")

### Model Summary

In [ ]:
model.summary()

## 5. Compile Model

In [ ]:
# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE']),
    loss='mse',
    metrics=['mae']
)

print("Model compiled successfully!")

## 6. Callbacks Setup

In [ ]:
# Define callbacks
model_name = f"{CONFIG['MODEL_TYPE']}_deblur_best.h5"
checkpoint_path = os.path.join(CONFIG['MODEL_SAVE_PATH'], model_name)

callbacks = [
    ModelCheckpoint(
        checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured:")
print("  - ModelCheckpoint (save best model)")
print("  - EarlyStopping (patience=10)")
print("  - ReduceLROnPlateau (patience=5)")

## 7. Train Model

In [ ]:
# Train model
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=CONFIG['BATCH_SIZE'],
    epochs=CONFIG['EPOCHS'],
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)

## 8. Training History Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# MAE plot
axes[1].plot(history.history['mae'], label='Training MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training and Validation MAE')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training history plots saved!")

## 9. Evaluate Model

In [ ]:
# Load best model
print(f"Loading best model from: {checkpoint_path}")
model = tf.keras.models.load_model(checkpoint_path)

# Make predictions on validation set
print("Making predictions on validation set...")
predictions = model.predict(X_val, batch_size=CONFIG['BATCH_SIZE'])

print(f"Predictions shape: {predictions.shape}")

### Calculate PSNR and SSIM

In [ ]:
# Calculate metrics
psnr_values = []
ssim_values = []

print("Calculating PSNR and SSIM metrics...")

for i in range(len(y_val)):
    # PSNR
    psnr_val = psnr(y_val[i], predictions[i], data_range=1.0)
    psnr_values.append(psnr_val)
    
    # SSIM
    ssim_val = ssim(y_val[i], predictions[i], data_range=1.0, channel_axis=2)
    ssim_values.append(ssim_val)

# Display results
print("\n" + "="*60)
print("EVALUATION METRICS")
print("="*60)
print(f"PSNR (Peak Signal-to-Noise Ratio):")
print(f"  Mean: {np.mean(psnr_values):.2f} dB")
print(f"  Std:  {np.std(psnr_values):.2f} dB")
print(f"\nSSIM (Structural Similarity Index):")
print(f"  Mean: {np.mean(ssim_values):.4f}")
print(f"  Std:  {np.std(ssim_values):.4f}")
print("="*60)

### Visualize Results

In [ ]:
# Visualize predictions
num_samples = min(5, len(X_val))
fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))

for i in range(num_samples):
    # Blurred input
    axes[i, 0].imshow(X_val[i])
    axes[i, 0].set_title("Blurred Input")
    axes[i, 0].axis('off')
    
    # Deblurred output
    axes[i, 1].imshow(predictions[i])
    axes[i, 1].set_title(f"Deblurred Output\nPSNR: {psnr_values[i]:.2f} dB")
    axes[i, 1].axis('off')
    
    # Ground truth
    axes[i, 2].imshow(y_val[i])
    axes[i, 2].set_title(f"Ground Truth\nSSIM: {ssim_values[i]:.4f}")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/deblur_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Results visualization saved!")

## 10. Save Final Model

In [ ]:
# Save final model
final_model_path = os.path.join(CONFIG['MODEL_SAVE_PATH'], f"{CONFIG['MODEL_TYPE']}_deblur_final.h5")
model.save(final_model_path)

print(f"\nFinal model saved to: {final_model_path}")
print(f"Best model saved to: {checkpoint_path}")

# Save model in SavedModel format (for deployment)
saved_model_dir = os.path.join(CONFIG['MODEL_SAVE_PATH'], f"{CONFIG['MODEL_TYPE']}_savedmodel")
model.save(saved_model_dir, save_format='tf')
print(f"SavedModel format saved to: {saved_model_dir}")

## 11. Export Metrics Summary

In [ ]:
# Save metrics to file
import json

metrics_summary = {
    'model_type': CONFIG['MODEL_TYPE'],
    'total_epochs': len(history.history['loss']),
    'final_train_loss': float(history.history['loss'][-1]),
    'final_val_loss': float(history.history['val_loss'][-1]),
    'psnr_mean': float(np.mean(psnr_values)),
    'psnr_std': float(np.std(psnr_values)),
    'ssim_mean': float(np.mean(ssim_values)),
    'ssim_std': float(np.std(ssim_values)),
    'num_train_samples': len(X_train),
    'num_val_samples': len(X_val)
}

metrics_path = os.path.join(CONFIG['MODEL_SAVE_PATH'], 'metrics_summary.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics_summary, f, indent=4)

print(f"Metrics summary saved to: {metrics_path}")
print("\nTraining complete! All outputs saved.")

## 12. Test on New Image (Optional)

Use this cell to test the model on a custom blurred image

In [ ]:
# Function to deblur a single image
def deblur_image(model, image_path, img_size):
    """
    Deblur a single image using the trained model.
    """
    # Load and preprocess
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, img_size, interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    
    # Add batch dimension
    img_batch = np.expand_dims(img, axis=0)
    
    # Predict
    deblurred = model.predict(img_batch, verbose=0)
    
    return img, deblurred[0]

# Example usage (uncomment to use with your own image)
# test_image_path = '/kaggle/input/your-test-image.jpg'
# blurred, deblurred = deblur_image(model, test_image_path, img_size)
# 
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# axes[0].imshow(blurred)
# axes[0].set_title('Blurred Input')
# axes[0].axis('off')
# axes[1].imshow(deblurred)
# axes[1].set_title('Deblurred Output')
# axes[1].axis('off')
# plt.show()

print("Test function ready! Upload an image and uncomment the code above to test.")